## Examen Módulo 4
## Alumno: Salvador Calderón Martínez

## 1. Ingeniería de Datos, Integración (Join) y Publicación en la Nube (Parte 1)

In [ ]:
import pandas as pd

## 1.1 Cargar datos


In [ ]:
personal = pd.read_csv("https://raw.githubusercontent.com/anaepm/rep/refs/heads/main/nobel_personal.csv")
personal


In [ ]:
premios = pd.read_csv("https://raw.githubusercontent.com/anaepm/rep/refs/heads/main/nobel_data.csv")
premios

# 1.2 Análisis de datos

## 1.2.1 Datos nulos

In [ ]:
print(personal.isna().sum())

In [ ]:
print(premios.isna().sum())

## 1.2.2 Datos duplicados

In [ ]:
print(premios.duplicated().sum())

In [ ]:
print(personal.duplicated().sum())

In [ ]:
personal["Laureate_Id"].duplicated().sum()

In [ ]:
premios["Laureate_Id"].duplicated().sum()


## 1.3 Limpieza

In [ ]:
# Apellido vacío se le agrega "unknown" para no perder información
personal["Lastname"] = personal["Lastname"].fillna("unknown")
premios["Lastname"] = premios["Lastname"].fillna("unknown")

# Quitar espacios
personal["Firstname"] = personal["Firstname"].str.strip()
personal["Lastname"] = personal["Lastname"].str.strip()
premios["Firstname"] = premios["Firstname"].str.strip()
premios["Lastname"] = premios["Lastname"].str.strip()

In [ ]:
# Elimnar duplicados en el id principal de personal
personal = personal.drop_duplicates(subset="Laureate_Id")
personal.shape

In [ ]:
# Elimine la columna con Laureate_Id de la tabla persona por que cuando
# la uni entra en conflicto con la misma columna de la tabla premio

personal = personal.drop(columns="Laureate_Id")

In [ ]:
# Las fechas estan en diferentes formatos, se arreglaran para cuando se usen en PowerBI.
personal["Birth_Year"] = personal["Birth_Date"].str.extract(r"(\d{4})").astype(float)
personal["Death_Year"] = personal["Death_Date"].str.extract(r"(\d{4})").astype(float)

# Algunos valores de año de nacimiento y muerte son 0, le asigne None
personal.loc[personal["Birth_Year"] == 0, "Birth_Year"] = None
personal.loc[personal["Death_Year"] == 0, "Death_Year"] = None


## 1.4 Realizar el Join

In [ ]:
nobel = premios.merge(personal, on=["Firstname", "Lastname"], how="left")
nobel.head()

In [ ]:
nobel.to_csv("nobelsalvador_limpio.csv", index=False)

In [ ]:
nobel.columns

In [ ]:
nobel.info

In [ ]:
# Mostrar archivo raw
pd.read_csv("https://raw.githubusercontent.com/SalvadorCM786/ExamenModuloIV/refs/heads/main/nobelsalvador_limpio.csv")

## 2. Procesamiento de Lenguaje Natural (ETL de Texto y Visualización)

In [ ]:
!pip install wordcloud -q

In [ ]:

import re
import matplotlib.pyplot as plt
from collections import Counter
from wordcloud import WordCloud, STOPWORDS

premios = pd.read_csv("https://raw.githubusercontent.com/SalvadorCM786/ExamenModuloIV/refs/heads/main/nobelsalvador_limpio.csv")
premios.groupby("Category")["Motivation"].describe()

In [ ]:
# verificar campos vacios en Motivation
print("Campos vacios en motivación:", premios["Motivation"].isna().sum())
print(premios["Category"].value_counts())


## 3. Limpieza y normalización del texto

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stopwords_en = set(STOPWORDS) | set(ENGLISH_STOP_WORDS)

def limpiar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r"[^a-z\s]", " ", texto)   
    texto = re.sub(r"\s+", " ", texto).strip()
    palabras = [w for w in texto.split() if w not in stopwords_en and len(w) > 2]
    return " ".join(palabras)

texto_df["Motivation_limpio"] = texto_df["Motivation"].apply(limpiar_texto)
texto_df[["Motivation", "Motivation_limpio"]].head()

## 4. Transformación: codificar `Category` como número

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
texto_df["Label"] = le.fit_transform(texto_df["Category"])

# qué número le tocó a cada categoría
dict(zip(le.classes_, le.transform(le.classes_)))

## 5. Vectorización (TF-IDF)

In [ ]:
vectorizador = TfidfVectorizer(
    stop_words="english",
    max_features=3000,
    min_df=2,          # ignora términos que aparecen en menos de 2 documentos (ruido)
    ngram_range=(1, 2) # incluye palabras individuales y pares de palabras
)

X = vectorizador.fit_transform(texto_df["Motivation_limpio"])
y = texto_df["Category"]

print("Matriz TF-IDF:", X.shape)
print("Ejemplo de términos:", vectorizador.get_feature_names_out()[:15])

In [ ]:
# nube de palabras
nube_completa = " ".join(texto_df["Motivation_limpio"])
print("Total de palabras:", len(nube_completa.split()))

# Frecuencia de palabras
frecuencias = Counter(nube_completa.split())


In [ ]:
# Nube de palabras 
nube = WordCloud(
    width=1200,
    height=700,
    background_color="white",
    stopwords=stopwords_en,
    colormap="viridis",
    max_words=150,
    collocations=False,
).generate(nube_completa)

plt.figure(figsize=(14, 8))
plt.imshow(nube, interpolation="bilinear")
plt.axis("off")
plt.title("Términos más frecuentes en las motivaciones de los Premios Nobel", fontsize=14)
plt.tight_layout()
plt.savefig("nube_palabras_nobel.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizador = TfidfVectorizer(max_features=3000, min_df=2)
X_tfidf = vectorizador.fit_transform(texto_df["Motivation_limpio"])
print("Matriz TF-IDF:", X_tfidf.shape)

# 3. Modelado Predictivo, Evaluación y Matriz de Confusión

## 6. Separar entrenamiento y prueba

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Entrenamiento:", X_train.shape, " Prueba:", X_test.shape)

## 7. Entrenar Naive Bayes

In [ ]:
modelo = MultinomialNB()
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
# Metricas
print(classification_report(y_test, y_pred, zero_division=0))

## 4. Desarrollo y Despliegue de la Aplicación Interactiva en Streamlit

In [ ]:
import numpy as np
import streamlit as st
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB


st.write(''' # Predicción de categoría de Premio Nobel ''')
st.image("Nobel.png", caption="Su creador fue el inventor sueco Alfred Nobel mediante su testamento en 1895.")

st.header('Texto')

def user_input_features():
  # Entrada
  texto = st.text_input("Introduce el texto a evaluar")

  user_input_data = {'Text': texto}

  features = pd.DataFrame(user_input_data, index=[0])

  return features

df = user_input_features()

nobel =  pd.read_csv('df_nobel.csv', encoding='latin-1')
X = nobel.Text
y = nobel.Label

vect = CountVectorizer()
X_dtm = vect.fit_transform(X)

nb = MultinomialNB()
nb.fit(X_dtm, y)

df_dtm = vect.transform(df['Text'])
prediction = nb.predict(df_dtm)

#{'physics':0, 'medicine':1, 'peace':2, 'literature':3, 'chemistry':4, 'economics':5}
#'Physics', 'Medicine', 'Peace', 'Literature', 'Chemistry', 'Economics'
st.subheader('Predicción')
if prediction == 0:
  st.write('Physics')
elif prediction == 1:
  st.write('Medicine')
elif prediction == 2:
  st.write('Peace')
elif prediction == 3:
  st.write('Literature')
elif prediction == 4:
  st.write('Chemistry')
elif prediction == 5:
  st.write('Economics')
else:
  st.write('Sin predicción')


In [ ]:
# Matriz de confusión
etiquetas = sorted(y.unique())
matriz = confusion_matrix(y_test, y_pred, labels=etiquetas)

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=matriz, display_labels=etiquetas)
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
ax.set_title("Matriz de confusión — Naive Bayes")
plt.tight_layout()
plt.savefig("matriz_confusion_nb.png", dpi=150)
plt.show()